In [1]:
!pip install xgboost
!pip install lightgbm


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


# 加载数据，并且合并数据集

In [2]:
import pandas as pd
# from sklearn.metrics import accuracy_score
import pickle
# from xgboost import XGBClassifier ,XGBRegressor
import pandas as pd
# from sklearn.metrics import accuracy_score
# from xgboost import XGBClassifier ,XGBRegressor
# from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, recall_score, precision_score
# from sklearn.metrics import precision_recall_curve, auc
# import lightgbm as lgb 
import joblib  
from sklearn.linear_model import Ridge,BayesianRidge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import BayesianRidge
from itertools import combinations
import xgboost as xgb

In [3]:

# from functools import reduce


# basics
basics =['administration_expense_ttm', 'asset_impairment_loss_ttm', 'cash_flow_to_price_ratio', 'circulating_market_cap', 'EBIT', 'EBITDA', 'financial_assets', 'financial_expense_ttm', 'financial_liability', 'goods_sale_and_service_render_cash_ttm', 'gross_profit_ttm', 'interest_carry_current_liability', 'interest_free_current_liability', 'market_cap', 'net_debt', 'net_finance_cash_flow_ttm', 'net_interest_expense', 'net_invest_cash_flow_ttm', 'net_operate_cash_flow_ttm', 'net_profit_ttm', 'net_working_capital', 'non_operating_net_profit_ttm', 'non_recurring_gain_loss', 'np_parent_company_owners_ttm', 'OperateNetIncome', 'operating_assets', 'operating_cost_ttm', 'operating_liability', 'operating_profit_ttm', 'operating_revenue_ttm', 'retained_earnings', 'sales_to_price_ratio', 'sale_expense_ttm', 'total_operating_cost_ttm', 'total_operating_revenue_ttm', 'total_profit_ttm', 'value_change_profit_ttm']
# emotion
emotion = ['AR', 'ARBR', 'ATR14', 'ATR6', 'BR', 'DAVOL10', 'DAVOL20', 'DAVOL5', 'MAWVAD', 'money_flow_20', 'PSY', 'turnover_volatility', 'TVMA20', 'TVMA6', 'TVSTD20', 'TVSTD6', 'VDEA', 'VDIFF', 'VEMA10', 'VEMA12', 'VEMA26', 'VEMA5', 'VMACD', 'VOL10', 'VOL120', 'VOL20', 'VOL240', 'VOL5', 'VOL60', 'VOSC', 'VR', 'VROC12', 'VROC6', 'VSTD10', 'VSTD20', 'WVAD']
# growth
growth = ['financing_cash_growth_rate', 'net_asset_growth_rate', 'net_operate_cashflow_growth_rate', 'net_profit_growth_rate', 'np_parent_company_owners_growth_rate', 'operating_revenue_growth_rate', 'PEG', 'total_asset_growth_rate', 'total_profit_growth_rate']
# momentum
momentum = ['arron_down_25', 'arron_up_25', 'BBIC', 'bear_power', 'BIAS10', 'BIAS20', 'BIAS5', 'BIAS60', 'bull_power', 'CCI10', 'CCI15', 'CCI20', 'CCI88', 'CR20', 'fifty_two_week_close_rank', 'MASS', 'PLRC12', 'PLRC24', 'PLRC6', 'Price1M', 'Price1Y', 'Price3M', 'Rank1M', 'ROC12', 'ROC120', 'ROC20', 'ROC6', 'ROC60', 'single_day_VPT', 'single_day_VPT_12', 'single_day_VPT_6', 'TRIX10', 'TRIX5', 'Volume1M']
# pershare
pershare = ['capital_reserve_fund_per_share', 'cashflow_per_share_ttm', 'cash_and_equivalents_per_share',
            'eps_ttm', 'net_asset_per_share', 'net_operate_cash_flow_per_share', 'operating_profit_per_share', 
            'operating_profit_per_share_ttm', 'operating_revenue_per_share', 'operating_revenue_per_share_ttm',
            'retained_earnings_per_share', 'retained_profit_per_share', 'surplus_reserve_fund_per_share', 
            'total_operating_revenue_per_share', 'total_operating_revenue_per_share_ttm']
# quality
quality = ['ACCA', 'accounts_payable_turnover_days', 'accounts_payable_turnover_rate', 
        'account_receivable_turnover_days', 'account_receivable_turnover_rate', 'adjusted_profit_to_total_profit',
        'admin_expense_rate', 'asset_turnover_ttm', 'cash_rate_of_sales', 'cash_to_current_liability', 'cfo_to_ev',
        'current_asset_turnover_rate', 'current_ratio', 'debt_to_asset_ratio', 'debt_to_equity_ratio',
        'debt_to_tangible_equity_ratio', 'DEGM', 'DEGM_8y', 'DSRI', 'equity_to_asset_ratio', 
        'equity_to_fixed_asset_ratio', 'equity_turnover_rate', 'financial_expense_rate', 'fixed_assets_turnover_rate',
        'fixed_asset_ratio', 'GMI', 'goods_service_cash_to_operating_revenue_ttm', 'gross_income_ratio',
        'intangible_asset_ratio', 'inventory_turnover_days', 'inventory_turnover_rate', 
        'invest_income_associates_to_total_profit', 'long_debt_to_asset_ratio', 'long_debt_to_working_capital_ratio',
        'long_term_debt_to_asset_ratio', 'LVGI', 'margin_stability', 'maximum_margin', 'MLEV',
        'net_non_operating_income_to_total_profit', 'net_operate_cash_flow_to_asset', 
        'net_operate_cash_flow_to_net_debt', 'net_operate_cash_flow_to_operate_income', 
        'net_operate_cash_flow_to_total_current_liability', 'net_operate_cash_flow_to_total_liability',
        'net_operating_cash_flow_coverage', 'net_profit_ratio', 'net_profit_to_total_operate_revenue_ttm',
        'non_current_asset_ratio', 'OperatingCycle', 'operating_cost_to_operating_revenue_ratio', 
        'operating_profit_growth_rate', 'operating_profit_ratio', 'operating_profit_to_operating_revenue',
        'operating_profit_to_total_profit', 'operating_tax_to_operating_revenue_ratio_ttm', 'profit_margin_ttm',
        'quick_ratio', 'rnoa_ttm', 'ROAEBITTTM', 'roa_ttm', 'roa_ttm_8y', 'roe_ttm', 'roe_ttm_8y', 'roic_ttm',
        'sale_expense_to_operating_revenue', 'SGAI', 'SGI', 'super_quick_ratio', 'total_asset_turnover_rate', 
        'total_profit_to_cost_ratio']
# risk
risk = ['Kurtosis120', 'Kurtosis20', 'Kurtosis60', 'sharpe_ratio_120', 'sharpe_ratio_20', 'sharpe_ratio_60', 'Skewness120', 'Skewness20', 'Skewness60', 'Variance120', 'Variance20', 'Variance60']
# style
style = ['average_share_turnover_annual', 'average_share_turnover_quarterly', 'beta', 'book_leverage', 'book_to_price_ratio', 'cash_earnings_to_price_ratio', 'cube_of_size', 'cumulative_range', 'daily_standard_deviation', 'debt_to_assets', 'earnings_growth', 'earnings_to_price_ratio', 'earnings_yield', 'growth', 'historical_sigma', 'leverage', 'liquidity', 'long_term_predicted_earnings_growth', 'market_leverage', 'momentum', 'natural_log_of_market_cap', 'non_linear_size', 'predicted_earnings_to_price_ratio', 'raw_beta', 'relative_strength', 'residual_volatility', 'sales_growth', 'share_turnover_monthly', 'short_term_predicted_earnings_growth', 'size']
# technical
technical =['boll_down', 'boll_up', 'EMA5', 'EMAC10', 'EMAC12', 'EMAC120', 'EMAC20', 'EMAC26', 'MAC10', 'MAC120', 'MAC20', 'MAC5', 'MAC60', 'MACDC', 'MFI14', 'price_no_fq']

jqfactors_list=basics + emotion + growth+momentum+pershare+quality+risk+style+technical

# # 读取数据
# X_basics_train  = pd.read_csv('train_baseline_2.0_basic.csv').fillna(0)
# X_basics_test   = pd.read_csv('test_baseline_2.0_basic.csv').fillna(0)
# X_emotion_train = pd.read_csv('train_baseline_2.0_emotion.csv').fillna(0)
# X_emotion_test  = pd.read_csv('test_baseline_2.0_emotion.csv').fillna(0)
# X_growth_train = pd.read_csv('train_baseline_2.0_growth.csv').fillna(0)
# X_growth_test  = pd.read_csv('test_baseline_2.0_growth.csv').fillna(0)
# X_momentum_train = pd.read_csv('train_baseline_2.0_momentum.csv').fillna(0)
# X_momentum_test  = pd.read_csv('test_baseline_2.0_momentum.csv').fillna(0)
# X_pershare_train = pd.read_csv('train_baseline_2.0_pershare.csv').fillna(0)
# X_pershare_test  = pd.read_csv('test_baseline_2.0_pershare.csv').fillna(0)
# X_quality_train = pd.read_csv('train_baseline_2.0_quality.csv').fillna(0)
# X_quality_test  = pd.read_csv('test_baseline_2.0_quality.csv').fillna(0)
# X_risk_train = pd.read_csv('train_baseline_2.0_risk.csv').fillna(0)
# X_risk_test  = pd.read_csv('test_baseline_2.0_risk.csv').fillna(0)
# X_style_train = pd.read_csv('train_baseline_2.0_style.csv').fillna(0)
# X_style_test  = pd.read_csv('test_baseline_2.0_style.csv').fillna(0)
# X_technical_train = pd.read_csv('train_baseline_2.0_technical.csv').fillna(0)
# X_technical_test  = pd.read_csv('test_baseline_2.0_technical.csv').fillna(0)


# # 合并数据
# dfs_train = [X_basics_train, X_emotion_train, X_growth_train, X_momentum_train,
#              X_pershare_train, X_quality_train, X_risk_train, X_style_train, X_technical_train]
# dfs_test = [X_basics_test, X_emotion_test, X_growth_test, X_momentum_test,
#             X_pershare_test, X_quality_test, X_risk_test, X_style_test, X_technical_test]

# X_train = reduce(lambda left, right: pd.merge(left, right, on=['Unnamed: 0', 'pchg', 'date'], how='inner'), dfs_train)
# X_test = reduce(lambda left, right: pd.merge(left, right, on=['Unnamed: 0', 'pchg', 'date'], how='inner'), dfs_test)



# # X_train = pd.merge(X_basics_train, X_emotion_train,X_growth_train,X_momentum_train,X_pershare_train,X_quality_train,X_risk_train,X_style_train,X_technical_train, on=['Unnamed: 0', 'pchg', 'date'], how='inner')
# # print(X_train.shape)

# # X_test = pd.merge(X_basics_test,  X_emotion_test,X_growth_test,X_momentum_test,X_pershare_test,X_quality_test,X_risk_test,X_style_test,X_technical_test, on=['Unnamed: 0', 'pchg', 'date'], how='inner')
# # print(X_test.shape)

# # 如果需要，可以重置索引
# X_train.reset_index(drop=True, inplace=True)
# X_test .reset_index(drop=True, inplace=True)

# # 把目标 y 提取出来
# y_train = X_basics_train['pchg']
# y_test  = X_basics_test['pchg']


# print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


# columns_to_fill = [col for col in jqfactors_list if col in X_train.columns]
# df_filled_train = X_train.copy()
# for column in columns_to_fill:
#     df_filled_train[column] = X_train.groupby('date')[column].transform(lambda x: x.fillna(x.median()))
# X_train = df_filled_train.dropna()

# df_filled_test = X_test.copy()
# for column in columns_to_fill:
#     df_filled_test[column] = X_test.groupby('date')[column].transform(lambda x: x.fillna(x.median()))
# X_test = df_filled_test.dropna()



# print(X_train.shape)
# print(X_test.shape)

# train_data = pd.concat([X_train, X_test], ignore_index=True)
# print(train_data.shape)


In [4]:
train_data = pd.read_csv('train_merged_all.csv').fillna(0)



In [5]:
factor_list = [
    (['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [-1e-10, 0.0001733817, -0.0022996607, -0.0024805333], 0.05),
    (['liquidity', 'roa_ttm'], [-0.0027762184, -0.002985997], 0.0),
    (['VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle'], [-5e-10, -6.874e-07, -0.0012473022, 3.958e-07], 0.0),
    (['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'], [7.38926e-05, -0.000365843, 0.0118442297], 0.0),
    (['TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5'], [-0.0, -0.0006079581, -0.0005131447, 5.775e-07, 0.130595625], -0.13),
]
##############################
factor_list = [
    (['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [-1e-10, 0.0001733817, -0.0022996607, -0.0024805333], 0.05),
    (['liquidity', 'roa_ttm'], [-0.0027762184, -0.002985997], 0.0),
    (['CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26'], [-6.8333e-06, -0.0018453753, 0.0082303157, 0.0417478679], -0.01),
    (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-5.3012e-05, 9.95318e-05, -1e-10], 0.0),
    (['debt_to_assets', 'BIAS10', 'ROC120', 'leverage'], [0.0033734282, -0.0006543384, -2.85833e-05, -0.0015953258], -0.0),
]
##########################################

factor_list = [
    (['liquidity', 'roa_ttm'], [-0.0027762184, -0.002985997], 0.0),
    (['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [-1e-10, 0.0001733817, -0.0022996607, -0.0024805333], 0.05),
    (['TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5'], [-0.0, -0.0006079581, -0.0005131447, 5.775e-07, 0.130595625], -0.13),
    (['turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12'], [-0.1320166292, -0.0025479137, 5.228e-07, 0.0816834823], -0.08),
    (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-5.3012e-05, 9.95318e-05, -1e-10], 0.0),
]


factor_names = [item[0] for item in factor_list]
xx = []

for sublist in factor_names:
   xx.append(sublist)
   
xx.append(['boll_up', 'turnover_volatility', 'size', 'sale_expense_ttm', 'momentum'])
xx.append(['cashflow_per_share_ttm', 'EMAC20', 'TRIX5', 'net_asset_per_share', 'average_share_turnover_quarterly'])


print(xx)
    
print("######################################")
# 获取所有清理后的因子名
all_factors = []
for feature_names, _, _ in factor_list:
    all_factors.extend(feature_names)

# 打印所有因子名
print(all_factors)

[['liquidity', 'roa_ttm'], ['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], ['TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5'], ['turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12'], ['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], ['boll_up', 'turnover_volatility', 'size', 'sale_expense_ttm', 'momentum'], ['cashflow_per_share_ttm', 'EMAC20', 'TRIX5', 'net_asset_per_share', 'average_share_turnover_quarterly']]
######################################
['liquidity', 'roa_ttm', 'inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap', 'TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5', 'turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12', 'price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate']


In [6]:


# xx.append(['BIAS5', 'np_parent_company_owners_growth_rate', 'ROC6', 'MAC60'])
# xx.append(['natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26'])
# xx.append(['CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26'])
# xx.append(['natural_log_of_market_cap', 'EMAC10', 'EMAC20', 'net_profit_ratio'])
# xx.append(['BIAS60', 'CR20', 'PLRC6', 'Variance120'])
# xx.append(['VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down'])
# xx.append(['debt_to_assets', 'BIAS10', 'ROC120', 'leverage'])
# xx.append(['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'])
# xx.append(['liquidity', 'roa_ttm'])
# xx.append(['VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle'])
# xx.append(['operating_revenue_growth_rate', 'surplus_reserve_fund_per_share', 'VSTD20', 'net_operate_cash_flow_to_operate_income'])
# xx.append(['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'])
# xx.append(['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'])
# xx.append(['non_current_asset_ratio', 'admin_expense_rate', 'VOL20'])
# xx.append(['MAC5', 'MLEV', 'EMAC120', 'fifty_two_week_close_rank'])
# xx.append(['turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12'])

# ################250606#################
# xx.append(['TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5'])# 0.007905 0.000640 0.003256 0.000029
# xx.append(['VOL60', 'TVSTD20', 'BBIC', 'goods_service_cash_to_operating_revenue_ttm'])

# xx.append(['BIAS5', 'np_parent_company_owners_growth_rate', 'ROC6', 'MAC60'])
# xx.append(['MAC5', 'MLEV', 'EMAC120', 'fifty_two_week_close_rank'])
# xx.append(['natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26'])
# xx.append(['turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12'])
# xx.append(['CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26'])
# xx.append(['VOL60', 'TVSTD20', 'BBIC', 'goods_service_cash_to_operating_revenue_ttm'])
# xx.append(['TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5'])
# xx.append(['natural_log_of_market_cap', 'EMAC10', 'EMAC20', 'net_profit_ratio'])
# xx.append(['BIAS60', 'CR20', 'PLRC6', 'Variance120'])
# xx.append(['VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down'])
# xx.append(['debt_to_assets', 'BIAS10', 'ROC120', 'leverage'])
# xx.append(['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'])
# xx.append(['liquidity', 'roa_ttm'])
# xx.append(['non_current_asset_ratio', 'admin_expense_rate', 'VOL20'])
# xx.append(['VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle'])
# xx.append(['operating_revenue_growth_rate', 'surplus_reserve_fund_per_share', 'VSTD20', 'net_operate_cash_flow_to_operate_income'])
# xx.append(['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'])
# xx.append(['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'])

# ################################
# xx.append(['TVMA20', 'bear_power', 'MAC20', 'TRIX5'])
# xx.append(['net_operating_cash_flow_coverage', 'accounts_payable_turnover_days', 'average_share_turnover_quarterly', 'non_linear_size', 'bull_power'])



# xx_all = list(set([item for sublist in xx for item in sublist]))

# print(xx_all)

# 分层指标衡量

In [7]:
import pandas as pd

import matplotlib.pyplot as plt

def calculate_layered_scores(result_df, returns_df, num_layers=20):
    """
    计算分层收益率并返回得分，用于比较不同预测结果。
    """
    backtest_results = pd.DataFrame(columns=[f'Group_{i}' for i in range(1, num_layers + 1)])

    for date in result_df.index:
        # 获取当天的因子值和收益率
        factors = result_df.loc[date]
        returns = returns_df.loc[date]

        # 合并因子值和收益率到一个DataFrame中
        combined_data = pd.DataFrame({'Factor': factors, 'Returns': returns})

        # 按因子值排序
        combined_data = combined_data.sort_values(by='Factor', ascending=False)

        # 分成num_layers组，每组数量相等
        group_size = len(combined_data) // num_layers
        groups = [combined_data.iloc[i * group_size:(i + 1) * group_size] for i in range(num_layers)]

        # 计算每个组的累计收益率
        group_returns = [group['Returns'].sum() for group in groups]

        # 将每个组的累积收益率添加到结果DataFrame中
        backtest_results.loc[date] = group_returns

    # 计算多空策略收益率（即顶层减去底层）
    long_short_returns = backtest_results[f'Group_1'] - backtest_results[f'Group_{num_layers}']

    # 计算得分：可根据需求修改，例如取累计收益率、平均收益率等
    score = long_short_returns.sum()

    return score


def calculate_layered_scores2(result_df, returns_df, num_layers=20):
    """
    计算分层收益率并返回得分，用于比较不同预测结果。
    修改说明：确保正确分组并计算平均收益率，保证顶层收益最优。
    """
    backtest_results = pd.DataFrame(columns=[f'Group_{i}' for i in range(1, num_layers + 1)])

    for date in result_df.index:
        # 获取当天的因子值和收益率
        factors = result_df.loc[date]
        returns = returns_df.loc[date]

        # 合并因子值和收益率到一个DataFrame中
        combined_data = pd.DataFrame({'Factor': factors, 'Returns': returns})

        # 按因子值降序排序（假设因子值越高收益越高）
        combined_data = combined_data.sort_values(by='Factor', ascending=False)

        # 分成num_layers组，正确处理余数以确保所有股票被分组
        n = len(combined_data)
        group_size = n // num_layers
        remainder = n % num_layers
        groups = []
        start = 0
        for i in range(num_layers):
            # 前remainder组每组多分配一个股票
            end = start + group_size + (1 if i < remainder else 0)
            group = combined_data.iloc[start:end]
            groups.append(group)
            start = end

        # 计算每个组的平均收益率（确保组间可比性）
        group_returns = [group['Returns'].mean() for group in groups]

        # 记录各组的平均收益率
        backtest_results.loc[date] = group_returns

    # 计算多空策略收益率（顶层平均收益 - 底层平均收益）
    long_short_returns = backtest_results['Group_1'] - backtest_results[f'Group_{num_layers}']

    # 得分：多空策略累计收益
    score = long_short_returns.sum()

    return score

def calculate_layered_scores3(result_df, returns_df, num_layers=100, smooth_window=10):
    """
    计算稳定的顶层收益，确保顶层收益稳定，适用于层数很多（如100层）的情况。
    修改说明：
    1. 适应更多层数的分组处理。
    2. 对顶层收益进行平滑（移动平均），提高收益的稳定性。
    """
    backtest_results = pd.DataFrame(columns=[f'Group_{i}' for i in range(1, num_layers + 1)])

    for date in result_df.index:
        # 获取当天的因子值和收益率
        factors = result_df.loc[date]
        returns = returns_df.loc[date]

        # 合并因子值和收益率到一个DataFrame中
        combined_data = pd.DataFrame({'Factor': factors, 'Returns': returns})

        # 按因子值降序排序（假设因子值越高收益越高）
        combined_data = combined_data.sort_values(by='Factor', ascending=False)

        # 分成num_layers组，正确处理余数以确保所有股票被分组
        n = len(combined_data)
        group_size = n // num_layers
        remainder = n % num_layers
        groups = []
        start = 0
        for i in range(num_layers):
            # 前remainder组每组多分配一个股票
            end = start + group_size + (1 if i < remainder else 0)
            group = combined_data.iloc[start:end]
            groups.append(group)
            start = end

        # 计算每个组的平均收益率（确保组间可比性）
        group_returns = [group['Returns'].mean() for group in groups]

        # 记录各组的平均收益率
        backtest_results.loc[date] = group_returns

    # 获取顶层收益（Group_1）
    top_layer_returns = backtest_results['Group_1']

    # 对顶层收益进行平滑处理（使用滑动平均平滑波动）
    smoothed_top_layer_returns = top_layer_returns.rolling(window=smooth_window, min_periods=1).mean()

    # 计算最终得分：使用平滑后的顶层收益
    score = smoothed_top_layer_returns.sum()

    return score


def calculate_layered_scores4(result_df, returns_df, num_layers=100, smooth_window=10):
    """
    在层数很多时确保顶层收益表现好。通过调整分层方式、给顶层更高的权重等手段，减少噪声。
    """
    backtest_results = pd.DataFrame(columns=[f'Group_{i}' for i in range(1, num_layers + 1)])

    for date in result_df.index:
        # 获取当天的因子值和收益率
        factors = result_df.loc[date]
        returns = returns_df.loc[date]

        # 合并因子值和收益率到一个DataFrame中
        combined_data = pd.DataFrame({'Factor': factors, 'Returns': returns})

        # 排序，假设因子值越高收益越高
        combined_data = combined_data.sort_values(by='Factor', ascending=False)

        # 根据分布调整分层策略：避免过于均匀的分层，确保每层有足够的差异
        n = len(combined_data)
        group_size = n // num_layers
        remainder = n % num_layers
        groups = []
        start = 0
        for i in range(num_layers):
            end = start + group_size + (1 if i < remainder else 0)
            group = combined_data.iloc[start:end]
            groups.append(group)
            start = end

        # 计算每组的平均收益率
        group_returns = [group['Returns'].mean() for group in groups]

        # 记录各组的收益率
        backtest_results.loc[date] = group_returns

    # 获取顶层收益（Group_1）并平滑
    top_layer_returns = backtest_results['Group_1']
    smoothed_top_layer_returns = top_layer_returns.rolling(window=smooth_window, min_periods=1).mean()

    # 对顶层收益加权：确保顶层收益的贡献更大
    weighted_top_layer_returns = smoothed_top_layer_returns * 1.5  # 例如，给顶层更高的权重

    # 计算最终得分：平滑后顶层收益 + 给顶层加权的收益
    score = weighted_top_layer_returns.sum()

    return score

def calculate_score(y_pred_proba, X_test, calculate_layered_scores4):
    """
    Calculate the score based on the predicted probabilities (y_pred_proba).

    Args:
    - y_pred_proba (pd.Series): Predicted probabilities from the model.
    - X_test (pd.DataFrame): DataFrame containing the test data, including 'id', 'date', and 'pchg'.
    - calculate_layered_scores4 (function): Function to calculate the score from the factor and return series.

    Returns:
    - score (float): The calculated score based on the predicted probabilities.
    """
    # Add the predicted probabilities as a factor column
    X_test['factor'] = y_pred_proba
    
    # Create the factor and return (ret) dataframes
    factor = X_test.set_index(['id', 'date'])['factor'].unstack(level='id')
    ret = X_test.set_index(['id', 'date'])['pchg'].unstack(level='id')

    # Calculate the score using the provided function
    score = calculate_layered_scores4(factor, ret, num_layers=100)
    
    return score


# 计算各个 predictions 的得分
# scores = []

# for i in range(len(predictions)):
#     y_pred_proba = predictions[i]
#     X_test['factor'] = y_pred_proba
#     factor = X_test.set_index(['id', 'date'])['factor']
#     factor = factor.unstack(level='id')
#     ret = X_test.set_index(['id', 'date'])['pchg']
#     ret = ret.unstack(level='id')

#     # 计算当前预测结果的得分
#     score = calculate_layered_scores4(factor, ret, num_layers=100)
#     scores.append(score)

# scores = []
# for i in range(len(predictions)):
#     y_pred_proba = predictions[i]
    
#     # Calculate score for each set of predicted probabilities
#     score = calculate_score(y_pred_proba, X_test, calculate_layered_scores4)
#     scores.append(score)






# sorted_scores = sorted(enumerate(scores, start=1), key=lambda x: x[1], reverse=True)

# new_predictions =[]
# max =5
# i = 0
# print("Models sorted by performance:")
# for idx, score in sorted_scores:
#     print(f"{xx[idx-1]}")
#     print(f"Model {idx}: Score = {score}")
#     if i < max:
#         new_predictions.append(predictions[idx-1])
#         i = i + 1

# 自动分数据集，端到端的比指标

In [8]:
# train_data
train_data.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
train_data['id']


0         002001.XSHE
1         002003.XSHE
2         002004.XSHE
3         002005.XSHE
4         002006.XSHE
             ...     
616112    003037.XSHE
616113    003038.XSHE
616114    003039.XSHE
616115    003040.XSHE
616116    003816.XSHE
Name: id, Length: 616117, dtype: object

In [9]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import BayesianRidge
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import r2_score, mean_squared_error

# # 假设 df 是你完整的数据，包含 target 和所有可能因子列
# # df = pd.read_csv('your_data.csv')

# # 你的因子组合列表（xx）
# # xx= []
# # xx.append(['natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26'])
# # xx.append(['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'])
# # xx.append(['goods_sale_and_service_render_cash_ttm', 'financial_expense_ttm', 'sale_expense_ttm', 'sales_to_price_ratio', 'VMACD'])
# # xx.append(['CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26'])
# # xx.append(['rnoa_ttm', 'VOL240', 'EMAC26', 'natural_log_of_market_cap', 'super_quick_ratio'])# 0.003817 0.061781  0.025288 0.041790
# # xx.append(['BIAS60', 'CR20', 'financial_assets', 'PLRC6', 'Variance120'] )#0.003833 0.061913  0.021114 0.041663
# # xx.append(['debt_to_assets', 'BIAS10', 'ROC120', 'leverage'] )#0.003839 0.061962  0.019563 0.041569
# # xx.append(['VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down'])

# # all_indicators = list(set([item for sublist in xx for item in sublist]))

# # print(all_indicators)


# results = []

# # 每组尝试 N 次随机划分
# N = 10
# test_size = 0.2



# for i, factor_group in enumerate(xx):
#     print(f"🔁 回测第{i+1}组因子: {factor_group}")
    
#     df_sub = train_data.dropna(subset=factor_group + ['pchg'])

#     # Check if data is sufficient
#     if len(df_sub) < 100:
#         print("❌ 数据不足，跳过该因子组")
#         continue

#     X = df_sub[['id', 'date','pchg'] + factor_group]
#     y = df_sub['pchg']

#     r2_list, mse_list, scores = [], [], []

#     for j in range(N):
#         X_train, X_test, y_train, y_test = train_test_split(
#             X, y, test_size=test_size, random_state=j
#         )
#         X_train0=X_train[factor_group]
#         X_test0=X_test[factor_group]



#         model = BayesianRidge()
#         model.fit(X_train0, y_train)
#         y_pred = model.predict(X_test0)

#         scores.append(calculate_score(y_pred, X_test, calculate_layered_scores4))

#         r2_list.append(r2_score(y_test, y_pred))
#         mse_list.append(mean_squared_error(y_test, y_pred))

#     results.append({
#         'factor_group_index': i + 1,
#         'factor_names': factor_group,
#         'avg_r2': np.mean(r2_list),
#         'std_r2': np.std(r2_list),
#         'avg_mse': np.mean(mse_list),
#         'std_mse': np.std(mse_list),
#         'avg_score': np.mean(scores),
#         'std_score': np.std(scores),
#     })

# # 转为 DataFrame 排序展示
# results_df = pd.DataFrame(results)
# results_df = results_df.sort_values(by=['avg_r2'], ascending=False)

# print("\n📊 因子组稳定性分析（按 avg_r2 排序）:")
# # print(results_df[['factor_group_index', 'factor_names', 'avg_r2', 'std_r2', 'avg_mse', 'std_mse','avg_score','std_score']])
# new_df = results_df[['factor_group_index', 'factor_names', 'avg_r2', 'std_r2', 'avg_mse', 'std_mse', 'avg_score', 'std_score']]
# new_df

In [10]:


# new_df = results_df[['factor_group_index', 'factor_names', 'avg_r2', 'std_r2', 'avg_mse', 'std_mse', 'avg_score', 'std_score']]

# new_df

# for item in new_df['factor_names']:
#     line = "xx.append([" + ", ".join(f"'{x}'" for x in item) + "])"
#     print(line)




In [11]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# from sklearn.linear_model import BayesianRidge
# from itertools import combinations

# # basics.extend(emotion)
# # jqfactors_list = basics

# # print(xx_all)
# notin = all_factors#['BIAS5', 'np_parent_company_owners_growth_rate', 'ROC6', 'MAC60', 'natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26', 'CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26', 'natural_log_of_market_cap', 'EMAC10', 'EMAC20', 'net_profit_ratio', 'BIAS60', 'CR20', 'PLRC6', 'Variance120', 'VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down', 'debt_to_assets', 'BIAS10', 'ROC120', 'leverage', 'inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap', 'liquidity', 'roa_ttm', 'VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle', 'operating_revenue_growth_rate', 'surplus_reserve_fund_per_share', 'VSTD20', 'net_operate_cash_flow_to_operate_income', 'super_quick_ratio', 'cube_of_size', 'cfo_to_ev', 'price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate', 'non_current_asset_ratio', 'admin_expense_rate', 'VOL20']

# filtered_list = [item for item in jqfactors_list if item not in notin]


# import random
# np.random.seed(42)
# seed = 0
# xx = []
# for i in range(9000):
#     random_integer = random.randint(2, 5)
#     random_sample = random.sample(filtered_list, random_integer)
#     xx.append(random_sample)
#     # print(random_sample)


# results=[]
# f=0
# for i in xx:
#     f=f+1
#     factor_group=list(i)
#     # X_train0 = X_train[jqfactors_list0]
#     # X_test0 = X_test[jqfactors_list0]


#     # print(f"🔁 回测因子: {factor_group}")
    
#     df_sub = train_data.dropna(subset=factor_group + ['pchg'])
#     if len(df_sub) < 100:
#         print("❌ 数据不足，跳过该因子组")
#         continue

#     X = df_sub[factor_group]
#     y = df_sub['pchg']

#     r2_list, mse_list = [], []

#     for j in range(N):
#         X_train, X_test, y_train, y_test = train_test_split(
#             X, y, test_size=test_size, random_state=j
#         )

#         model = BayesianRidge()
#         model.fit(X_train, y_train)
#         y_pred = model.predict(X_test)

#         r2_list.append(r2_score(y_test, y_pred))
#         mse_list.append(mean_squared_error(y_test, y_pred))

#     results.append({
#         'factor_names': factor_group,
#         'avg_r2': np.mean(r2_list),
#         'std_r2': np.std(r2_list),
#         'avg_mse': np.mean(mse_list),
#         'std_mse': np.std(mse_list)
#     })

#     if (f % 1000) == 0:
#         print(f)
#         results_df = pd.DataFrame(results)
#         sorted_results_df = results_df.sort_values(by=['avg_r2'], ascending=False)
#         print(sorted_results_df.head(10).to_string(index=False))
    
# # 转换为 DataFrame
# results_df = pd.DataFrame(results)

# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)

# sorted_results_df = results_df.sort_values(by='avg_r2', ascending=False)
# print(sorted_results_df.head(100).to_string(index=False))

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import BayesianRidge
from itertools import combinations

# basics.extend(emotion)
jqfactors_list = [
    'average_share_turnover_annual',
    'average_share_turnover_quarterly',
    'beta',
    'book_leverage',
    'book_to_price_ratio',
    'cash_earnings_to_price_ratio',
    'cube_of_size',
    'cumulative_range',
    'daily_standard_deviation',
    'debt_to_assets',
    'earnings_growth',
    'earnings_to_price_ratio',
    'earnings_yield',
    'growth',
    'historical_sigma',
    'leverage',
    'liquidity',
    'long_term_predicted_earnings_growth',
    'market_leverage',
    'momentum',
    'natural_log_of_market_cap',
    'non_linear_size',
    'predicted_earnings_to_price_ratio',
    'raw_beta',
    'relative_strength',
    'residual_volatility',
    'sales_growth',
    'share_turnover_monthly',
    'short_term_predicted_earnings_growth',
    'size'
]

# print(xx_all)
notin = all_factors#['BIAS5', 'np_parent_company_owners_growth_rate', 'ROC6', 'MAC60', 'natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26', 'CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26', 'natural_log_of_market_cap', 'EMAC10', 'EMAC20', 'net_profit_ratio', 'BIAS60', 'CR20', 'PLRC6', 'Variance120', 'VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down', 'debt_to_assets', 'BIAS10', 'ROC120', 'leverage', 'inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap', 'liquidity', 'roa_ttm', 'VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle', 'operating_revenue_growth_rate', 'surplus_reserve_fund_per_share', 'VSTD20', 'net_operate_cash_flow_to_operate_income', 'super_quick_ratio', 'cube_of_size', 'cfo_to_ev', 'price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate', 'non_current_asset_ratio', 'admin_expense_rate', 'VOL20']

filtered_list = [item for item in jqfactors_list if item not in notin]


import random
np.random.seed(42)
seed = 0
xx = []
for i in range(100):
    random_integer = random.randint(2, 5)
    random_sample = random.sample(filtered_list, random_integer)
    xx.append(random_sample)
    print(random_sample)


train_window = 2520  # 初始训练窗口长度
test_window = 252    # 测试窗口长度

results = []
f = 0

for i in xx:
    f += 1
    factor_group = list(i)

    df_sub = train_data.dropna(subset=factor_group + ['pchg']).copy()
    if len(df_sub) < train_window + test_window:
        print("❌ 数据不足，跳过该因子组")
        continue

    df_sub = df_sub.sort_index()  # 保证按时间顺序

    X_all = df_sub[factor_group]
    y_all = df_sub['pchg']

    r2_list, mse_list,r2_time_series,r2_time_stamps = [], [],[],[]

    # 扩展窗口：初始为 train_window，之后不断扩大训练集
    for end_train in range(train_window, len(df_sub) - test_window + 1, test_window):
        start_test = end_train
        end_test = start_test + test_window

        X_train = X_all.iloc[:end_train]
        y_train = y_all.iloc[:end_train]
        X_test = X_all.iloc[start_test:end_test]
        y_test = y_all.iloc[start_test:end_test]

        model = BayesianRidge()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        r2 = r2_score(y_test, y_pred)
        r2_list.append(r2)
        mse_list.append(mean_squared_error(y_test, y_pred))

        test_end_time = df_sub.index[end_test - 1]  # 获取测试窗口结束时间
        r2_time_series.append((test_end_time, r2))


        time_stamps = df_sub.index[start_test:end_test][-1]  # 获取测试窗口结束的时间
        r2_time_stamps.append((time_stamps, r2))


    if len(r2_list) == 0:
        continue


    results.append({
        'factor_names': factor_group,
        'avg_r2': np.mean(r2_list),
        'std_r2': np.std(r2_list),
        # 'r2_series': r2_list,
        # 'r2_timestamps': r2_time_stamps,
        'avg_mse': np.mean(mse_list),
        'std_mse': np.std(mse_list)
    })

    if f % 5 == 0:
        print(f"已完成 {f} 组因子回测")
        results_df = pd.DataFrame(results)
        sorted_results_df = results_df.sort_values(by='avg_r2', ascending=False)
        print(sorted_results_df.head(10).to_string(index=False))

sorted_results_df = results_df.sort_values(by='avg_r2', ascending=False)
print(sorted_results_df.head(100).to_string(index=False))

['market_leverage', 'historical_sigma', 'book_to_price_ratio']
['cumulative_range', 'average_share_turnover_quarterly', 'market_leverage', 'cube_of_size']
['raw_beta', 'residual_volatility', 'earnings_to_price_ratio']
['average_share_turnover_annual', 'size', 'leverage']
['book_to_price_ratio', 'book_leverage', 'momentum', 'long_term_predicted_earnings_growth', 'cash_earnings_to_price_ratio']
['market_leverage', 'sales_growth', 'earnings_yield']
['average_share_turnover_annual', 'raw_beta', 'cash_earnings_to_price_ratio', 'long_term_predicted_earnings_growth']
['average_share_turnover_annual', 'size', 'relative_strength', 'momentum']
['earnings_to_price_ratio', 'cumulative_range', 'leverage']
['predicted_earnings_to_price_ratio', 'sales_growth', 'raw_beta', 'average_share_turnover_quarterly']
['momentum', 'cube_of_size', 'historical_sigma']
['long_term_predicted_earnings_growth', 'sales_growth']
['earnings_growth', 'cash_earnings_to_price_ratio', 'size', 'earnings_yield']
['earnings_gr

KeyboardInterrupt: 

In [13]:
# ############并发版本
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.linear_model import BayesianRidge
# import random
# import concurrent.futures

# # 假设这些变量已在外部定义好：
# # train_data, all_factors, jqfactors_list

# # 1. 去除不在使用列表的因子
# filtered_list = [item for item in jqfactors_list if item not in all_factors]

# # 2. 构造 9000 个因子组合，每组 2~5 个
# np.random.seed(42)
# xx = []
# for i in range(10):
#     random_integer = random.randint(2, 5)
#     random_sample = random.sample(filtered_list, random_integer)
#     xx.append(random_sample)

# print(xx)

# # 3. 参数设置
# train_window = 252  # 初始训练窗口长度
# test_window = 20    # 测试窗口长度

# # 4. 定义计算函数（单组因子组合的回测逻辑）
# def evaluate_factor_group(factor_group):
#     df_sub = train_data.dropna(subset=factor_group + ['pchg']).copy()
#     if len(df_sub) < train_window + test_window:
#         return None

#     df_sub = df_sub.sort_index()  # 确保时间顺序
#     X_all = df_sub[factor_group]
#     y_all = df_sub['pchg']

#     r2_list, mse_list = [], []

#     for end_train in range(train_window, len(df_sub) - test_window + 1, test_window):
#         start_test = end_train
#         end_test = start_test + test_window

#         X_train = X_all.iloc[:end_train]
#         y_train = y_all.iloc[:end_train]
#         X_test = X_all.iloc[start_test:end_test]
#         y_test = y_all.iloc[start_test:end_test]

#         model = BayesianRidge()
#         model.fit(X_train, y_train)
#         y_pred = model.predict(X_test)

#         r2_list.append(r2_score(y_test, y_pred))
#         mse_list.append(mean_squared_error(y_test, y_pred))

#     if len(r2_list) == 0:
#         return None

#     return {
#         'factor_names': factor_group,
#         'avg_r2': np.mean(r2_list),
#         'std_r2': np.std(r2_list),
#         'avg_mse': np.mean(mse_list),
#         'std_mse': np.std(mse_list)
#     }

# # 5. 多进程执行主循环（等价于原逻辑）
# results = []
# with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
#     futures = [executor.submit(evaluate_factor_group, list(fg)) for fg in xx]

#     for idx, future in enumerate(concurrent.futures.as_completed(futures), 1):
#         res = future.result()
#         if res:
#             results.append(res)
        
#         if idx % 5 == 0:
#             print(f"已完成 {idx} 组因子回测")
#             results_df = pd.DataFrame(results)
#             sorted_results_df = results_df.sort_values(by='avg_r2', ascending=False)
#             print(sorted_results_df.head(10).to_string(index=False))

# # 6. 全部完成后输出最终前 100 名
# results_df = pd.DataFrame(results)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)
# sorted_results_df = results_df.sort_values(by='avg_r2', ascending=False)
# print(sorted_results_df.head(100).to_string(index=False))


KeyboardInterrupt: 

In [15]:
import re


data="""
[relative_strength, size, cube_of_size, predicted_earnings_to_price_ratio] -0.315029 0.557438 0.003276 0.002938
                          [residual_volatility, earnings_to_price_ratio, book_to_price_ratio, size] -0.315036 0.558264 0.003276 0.002930
[book_to_price_ratio, long_term_predicted_earnings_growth, predicted_earnings_to_price_ratio, size] -0.315390 0.556832 0.003277 0.002928
                                                    [average_share_turnover_annual, size, leverage] -0.315523 0.557589 0.003277 0.002926
                              [cube_of_size, historical_sigma, long_term_predicted_earnings_growth] -0.315606 0.556670 0.003278 0.002932
                       [earnings_growth, short_term_predicted_earnings_growth, residual_volatility] -0.315637 0.557793 0.003277 0.002934
                [cumulative_range, average_share_turnover_quarterly, market_leverage, cube_of_size] -0.315645 0.556384 0.003277 0.002926
                              [earnings_growth, cash_earnings_to_price_ratio, size, earnings_yield] -0.315663 0.557260 0.003278 0.002931
                                       [size, predicted_earnings_to_price_ratio, relative_strength] -0.315892 0.557591 0.003278 0.002938
                                                                 [cumulative_range, momentum, size] -0.315923 0.557480 0.003278 0.002931"""


lines = data.strip().split("\n")

seen_factors = set()
result_lines = []

for line in lines:
    match = re.match(r"(\[.*?\])\s+(.*)", line)
    if match:
        array_str, _ = match.groups()
        
        # 提取因子名列表
        factors = re.findall(r"\w+", array_str)
        
        if any(factor in seen_factors for factor in factors):
            continue
        
        seen_factors.update(factors)
        result_lines.append(factors)

# 按目标格式输出
for factors in result_lines:
    print(f"xx.append({factors})")

xx.append(['relative_strength', 'size', 'cube_of_size', 'predicted_earnings_to_price_ratio'])


In [7]:
import pandas as pd



# 重命名股票代码列（如果还没有）
df.rename(columns={'Unnamed: 0': 'stock_id'}, inplace=True)

# 要保留的字段
keep_cols = [
    'stock_id',     # 股票代码
    'date',         # 日期
    'pchg',         # 如果有这个字段
    'non_linear_size', 
    'beta',
    'book_to_price_ratio',
    'earnings_yield',
    'growth'
]

# 保留字段（过滤出存在的列）
keep_cols = [col for col in keep_cols if col in df.columns]

# 生成新 DataFrame 并保存
filtered_df = df[keep_cols]
filtered_df.to_csv('filtered_factors.csv', index=False)

print("✅ 已保存为 filtered_factors.csv")



✅ 已保存为 filtered_factors.csv


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5ffe6543-a1b2-431c-ba55-b0c1984cba7c' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>